In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
import pickle
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

#### Lectura de train y test

In [2]:
df = pd.read_csv(r'C:\Users\plaza\Desktop\Documentos_Clase\ONLINE_DS_THEBRIDGE_Alejandro_Plaza\Proyecto_ML\data\processed\dfdefinitivo.csv')
df.head()

,Hours_Studied,Attendance,Previous_Scores,Tutoring_Sessions,PI_High,PI_Low,AtR_High,AtR_Low,Exam_Score
0,0,4,7,1,0,0,0,1,9
1,1,27,13,0,0,1,0,0,3
2,1,25,6,1,1,0,0,0,9
3,1,13,11,1,0,0,0,0,2
4,2,23,1,1,0,1,1,0,7


#### Dividimos en train y test

In [3]:
X = df.drop(columns = 'Exam_Score')
poly = PolynomialFeatures(degree = 2)
X = poly.fit_transform(X)
y = df['Exam_Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 30)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(8000, 45)
(2000, 45)
(8000,)
(2000,)


### Buscamos el mejor alpha para Ridge

In [4]:
n_alphas = 100
alphas = np.logspace(-4, 3, n_alphas) 

coef_ridge = []
err_ridge = []

for a in alphas:
    ridge = Ridge(alpha=a)
    ridge.fit(X_train, y_train)
    
    coef_ridge.append(ridge.coef_)
    
    y_pred = ridge.predict(X_test)
    ridge_error = mean_squared_error(y_pred, y_test)
    
    err_ridge.append(ridge_error)

In [5]:
min(err_ridge)

23.226736039791422

In [6]:
alphas[np.array(err_ridge).argmin()]

np.float64(17.073526474706888)

## Entrenamos el modelo de Ridge con el mejor alpha

In [7]:
ridgeR = Ridge(alpha = 17.07)
ridgeR.fit(X_train, y_train)

print("MAE", mean_absolute_error(y_test, ridgeR.predict(X_test)))
print("MSE", mean_squared_error(y_test, ridgeR.predict(X_test)))
print("RMSE", mean_squared_error(y_test, ridgeR.predict(X_test)) ** (1/2))
print("MAPE", mean_absolute_percentage_error(y_test, ridgeR.predict(X_test)))
print("r2_score", r2_score(y_test, ridgeR.predict(X_test)))

MAE 3.910300079219069
MSE 23.226736040697226
RMSE 4.81941241653972
MAPE 347891391679872.2
r2_score 0.9720101118266105


### Buscamos el mejor alpha para Lasso

In [12]:
n_alphas = 100
alphas = np.logspace(-4, 3, n_alphas) 

coef_ridge = []
err_ridge = []

for a in alphas:
    ridge = Lasso(alpha=a)
    ridge.fit(X_train, y_train)
    
    coef_ridge.append(ridge.coef_)
    
    y_pred = ridge.predict(X_test)
    ridge_error = mean_squared_error(y_pred, y_test)
    
    err_ridge.append(ridge_error)

c:\Users\plaza\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.940e+04, tolerance: 6.705e+02
  model = cd_fast.enet_coordinate_descent(
c:\Users\plaza\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.940e+04, tolerance: 6.705e+02
  model = cd_fast.enet_coordinate_descent(
c:\Users\plaza\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the sca

In [9]:
min(err_ridge)

23.21422636033371

In [10]:
alphas[np.array(err_ridge).argmin()]

np.float64(0.005857020818056662)

## Entrenamos el modelo de Lasso con el mejor alpha

In [13]:
lassoR = Lasso(alpha=0.0058)
lassoR.fit(X_train, y_train)

print("MAE", mean_absolute_error(y_test, lassoR.predict(X_test)))
print("MSE", mean_squared_error(y_test, lassoR.predict(X_test)))
print("RMSE", mean_squared_error(y_test, lassoR.predict(X_test)) ** (1/2))
print("MAPE", mean_absolute_percentage_error(y_test, lassoR.predict(X_test)))
print("r2_score", r2_score(y_test, lassoR.predict(X_test)))

MAE 3.910841066882375
MSE 23.21423208018043
RMSE 4.818114992419798
MAPE 351566706463407.7
r2_score 0.9720251799987366


c:\Users\plaza\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.960e+04, tolerance: 6.705e+02
  model = cd_fast.enet_coordinate_descent(


#### Hacemos unas pruebas para Lasso con nuevos datos inventados, serían outlayers

In [15]:
lassoR.predict(poly.fit_transform([[0, 0, 73, 3, 1, 0, 0, 0]]))

array([30.53585753])

In [16]:
lassoR.predict(poly.fit_transform([[0, 100, 0, 3, 1, 0, 0, 0]]))

array([21.19761359])

In [17]:
lassoR.predict(poly.fit_transform([[20, 0, 0, 3, 1, 0, 0, 0]]))

array([30.84596919])

#### Hacemos unas pruebas para Ridge con nuevos datos inventados, serían outlayers

In [18]:
ridgeR.predict(poly.fit_transform([[0, 0, 73, 3, 1, 0, 0, 0]]))

array([30.29551354])

In [19]:
ridgeR.predict(poly.fit_transform([[0, 100, 0, 3, 1, 0, 0, 0]]))

array([8.71585646])

In [20]:
ridgeR.predict(poly.fit_transform([[20, 0, 0, 3, 1, 0, 0, 0]]))

array([28.38323072])

## Guardamos ambos modelos

In [21]:
with open(r"C:\Users\plaza\Desktop\Documentos_Clase\ONLINE_DS_THEBRIDGE_Alejandro_Plaza\Proyecto_ML\models\lassoR.pkl", "wb") as f:
    pickle.dump(lassoR, f)

In [22]:
with open(r"C:\Users\plaza\Desktop\Documentos_Clase\ONLINE_DS_THEBRIDGE_Alejandro_Plaza\Proyecto_ML\models\ridgeR.pkl", "wb") as f:
    pickle.dump(ridgeR, f)